# ReMDM MiniHack — Demo Notebook



This notebook is a fully self-contained demonstration of **Remasking Discrete Diffusion Models (ReMDM) for action-sequence planning in MiniHack navigation environments**. It accompanies the paper *Return-Weighted ELBO Fine-Tuning Degrades Masked Diffusion Planners*. We study what the standard tractable objective for aligning a masked discrete diffusion planner with a reward actually does: across 25 ablation conditions in MiniHack and Craftax Classic, three seeds each, it **makes the planner worse** — no condition on Craftax Classic recovers the checkpoint it started from — and an exact decomposition of its gradient shows that the return weighting is **not** what does the damage.

Running this notebook top-to-bottom in a fresh Google Colab runtime downloads the source code, the pre-trained DAgger checkpoint, and the pre-computed RL fine-tuning ablation assets from a public HuggingFace repository, then evaluates the model live on procedurally generated MiniHack layouts and visualises both the agent's behaviour and the underlying diffusion denoising process.

**How to use this notebook**

1. Open this notebook in Colab (`File > Upload notebook` > select `demo_minihack.ipynb`).
2. *(Optional)* Tweak the configuration constants in **Cell 1** below — change `ID_ENVS` / `OOD_ENVS` to test other registered environments, change `SEED` to test fresh procedurally-generated layouts, raise `EPISODES_PER_ENV` for tighter error bars, or set `CUSTOM_DES_FILE` to a hand-authored MiniHack `.des` level uploaded to Colab.
3. *(Optional)* `Runtime > Change runtime type > T4 GPU` for the fastest run.
4. `Runtime > Run all`.

**Runtime budget** (default `EPISODES_PER_ENV=20`): ≈10 min on a Colab T4 GPU, ≈25 min on CPU. The first cell installs NLE which compiles C code — this takes ~3 min by itself.

**Which evaluation path this notebook uses.** Cell 5 calls `Evaluator._run_episodes_batched` from `src/planners/inference.py` — the *same* evaluator the ablation harness calls (`experiments/rl_finetuning/ablations/training.py` imports `Evaluator` and calls `evaluator.evaluate(...)` at every eval point). The notebook and the paper's MiniHack numbers therefore come from one code path, with one sampler and one replanning rule. The only difference is episode count and seed offset.

**Reproducibility statement.** Training is done **offline** — this notebook only runs inference on the supplied checkpoint, so that the notebook stays runnable on a free Colab runtime and can be tested on unseen inputs. Because MiniHack environments are procedurally generated, every run with a different `SEED` exposes the model to layouts it has never seen, including in the four in-distribution maps it was trained on.

## Cell 1 — Configuration

Everything you may want to change lives in this single cell.

In [ ]:
# ============================================================================
# Configuration — change these to test on unseen inputs
# ============================================================================

# Public HuggingFace repo containing source code, the stripped pre-trained
# checkpoint, and the pre-computed ablation assets. No auth required.
# Point it at your own Hub repo holding the same layout, or train from
# source (see README.md).
HF_REPO_ID: str = "MathisW78/remdm-minihack"

# Reproducibility seed. The live evaluator generates env seeds as
# (SEED + episode_index) per environment, so changing this number tests the
# model on a fresh batch of procedurally generated layouts.
SEED: int = 42

# Episodes per environment for the live evaluation pass. Higher = tighter
# error bars but slower. The reported numbers in the paper used 50.
EPISODES_PER_ENV: int = 20

# In-distribution training environments (4 maps).
ID_ENVS: list[str] = [
    "MiniHack-Room-Random-5x5-v0",
    "MiniHack-Room-Random-15x15-v0",
    "MiniHack-Corridor-R2-v0",
    "MiniHack-MazeWalk-9x9-v0",
]

# Out-of-distribution zero-shot evaluation environments (3 maps).
OOD_ENVS: list[str] = [
    "MiniHack-Room-Dark-15x15-v0",
    "MiniHack-Corridor-R5-v0",
    "MiniHack-MazeWalk-45x19-v0",
]

# Optional path to a custom .des MiniHack scenario file uploaded to Colab.
# Leave as None to skip; set to e.g. "/content/my_level.des" to evaluate the
# model on a hand-authored MiniHack level alongside the registry envs.
CUSTOM_DES_FILE: str | None = None

# Inference device. None = auto-detect (CUDA if available, else CPU).
INFERENCE_DEVICE: str | None = None

# Local directory the HF snapshot is downloaded into.
SNAPSHOT_DIR: str = "remdm-minihack"

## Cell 2 — Setup & installation

The next three cells install NetHack Learning Environment (NLE), MiniHack, PyTorch, and the supporting libraries; verify that NLE compiled correctly; and download the project source + checkpoint from HuggingFace. **NLE is the highest-risk failure point on Colab** because it has to compile C code under the hood. If the verification cell fails with an import error, restart the Colab runtime (`Runtime > Restart runtime`) and re-run from the install cell.

In [ ]:
# ── 1. System dependencies + Python packages ─────────────────────────────
import os
import subprocess
import sys

print("[1/3] Installing system dependencies for NLE (NetHack)...")
try:
    _apt = subprocess.run(
        [
            "apt-get", "install", "-y", "-q",
            "cmake", "build-essential", "bison", "flex", "libbz2-dev",
        ],
        check=False, capture_output=True, text=True,
    )
    if _apt.returncode != 0:
        # Non-Colab environments (e.g. local Jupyter) will fail apt-get; that is
        # fine if the deps are already installed. Print and continue.
        print("  apt-get returned non-zero (likely already installed or non-Debian env):")
        print("  " + (_apt.stderr or "").strip()[-400:])
except FileNotFoundError:
    # No apt-get at all (macOS, non-Debian Linux). Fine if the build deps are
    # already present; NLE's compilation is verified in the next cell either way.
    print("  apt-get not found (non-Debian environment) - skipping system deps.")

print("[2/3] Installing NLE (compiles NetHack from source — slow first run)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nle>=1.2.0"],
    check=True,
)

print("[3/3] Installing MiniHack + supporting libraries...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "minihack>=1.0.2",
        "torch>=2.4",
        "huggingface_hub>=0.25",
        "polars>=1.0",
        "matplotlib>=3.8",
        "pyyaml>=6.0",
        "gymnasium>=0.29",
        "numpy>=1.26",
    ],
    check=True,
)
print("Install step complete.")

In [ ]:
# ── 2. Verify NLE/MiniHack actually loaded ───────────────────────────────
# Fail loudly with a clear message if the C compilation step did not work.
try:
    import nle  # noqa: F401
    import minihack  # noqa: F401
    import gymnasium as gym
    _env = gym.make(
        "MiniHack-Room-Random-5x5-v0",
        observation_keys=("glyphs", "chars"),
    )
    _obs, _info = _env.reset(seed=0)
    _env.close()
    print(f"NLE OK. Glyphs shape: {_obs['glyphs'].shape}")
except Exception as exc:
    raise RuntimeError(
        "NLE/MiniHack failed to import or instantiate. On Colab this almost "
        "always means the C build of NetHack did not finish. Please restart "
        "the runtime (Runtime > Restart runtime) and re-run the install "
        "cell from a clean state."
    ) from exc

In [ ]:
# ── 3. Download source + checkpoint + ablation assets from HuggingFace ──
from huggingface_hub import snapshot_download

if HF_REPO_ID == "UNSET_HF_REPO_ID":
    raise RuntimeError(
        "HF_REPO_ID is unset. Edit the constant in Cell 1 to point at a "
        "public HuggingFace repo containing the source + checkpoint + assets."
    )

snapshot_path = snapshot_download(repo_id=HF_REPO_ID, local_dir=SNAPSHOT_DIR)
print(f"Snapshot at: {snapshot_path}")

# Make `import src.*` resolve against the downloaded snapshot.
if snapshot_path not in sys.path:
    sys.path.insert(0, snapshot_path)

# Smoke imports — fail here if the snapshot is incomplete.
from src.config import load_config
from src.models.denoiser import make_model, ModelEMA
from src.diffusion.sampling import remdm_sample
from src.envs.minihack_env import make_env
from src.planners.inference import Evaluator, format_eval_results
print("Imports OK.")

## Cell 3 — Project overview

**Problem.** MiniHack navigation environments are sparse-reward gridworlds with procedurally generated dungeon layouts, locked doors, mazes, and partial observability. Off-the-shelf model-free reinforcement learning baselines (PPO, A2C, DQN, recurrent PPO) do not learn these layouts from scratch within our compute budget, reaching between 0.3% and 4.7% in-distribution after 5M environment steps at default hyperparameters — which is why we train by imitation. They are reported to establish that fact, not as a matched comparison.

**Approach.** We treat planning as **discrete masked diffusion over action sequences**. A dual-stream transformer denoiser, conditioned on (i) a 9×9 local glyph crop centred on the agent and (ii) the full 21×79 global dungeon map, generates 64-step action plans by iteratively denoising a sequence of `[MASK]` tokens. At inference time we use **ReMDM** (Remasking Discrete Diffusion Models — Wang et al. 2025): MaskGIT-style progressive unmasking interleaved with stochastic confidence-weighted **remasking**, which lets the model revise low-confidence commitments mid-trajectory rather than baking in early mistakes.

**Architecture (`LocalDiffusionPlannerWithGlobal`, ≈5.2M parameters, PyTorch).**

```
Local stream:    9×9 glyphs   → Embed(6000,64) → CNN(64→32→64) → Linear → 1 token  (256-D)
Global stream:   21×79 glyphs → Embed(6000,32) → CNN(32→32→64) → AdaptivePool(2,4)
                              → Linear(64,256) → 8 spatial tokens (256-D)
                              + auxiliary goal head: mean(global) → MLP → [B,2]  (normalised staircase coords)
                              × sigmoid(learnable scalar gate, init logit = −3.0)  ← keeps global stream nearly closed early in training
Action stream:   action_emb(14,256) + timestep_emb(100,256) + position_emb(64,256)
Transformer:     concat [local(1) + global(8) + actions(64) = 73 tokens]
                 → 4-layer bidirectional TransformerEncoder (256-D, 4 heads, GELU, pre-norm) → last 64 tokens
Action head:     Linear(256, 12) → action logits  (12 movement / interact actions; MASK=12 and PAD=13 are inputs only)
```

The auxiliary staircase-coordinate head **has no counterpart in the offline baselines**, and the paper lists it as one of four confounds in the imitation comparison (Cell 8).

**Diffusion process.** Forward: each token is independently replaced with `[MASK]` with probability σ(t) under a linear schedule. Reverse (inference): K=10 denoising steps with confidence-based progressive unmasking interleaved with stochastic remasking at probability p_remask = η · (1 − r), where η = 0.15 and r is the progress through the chain. DAgger collection uses greedy argmax sampling at K=5 steps. The planner runs in **receding horizon** mode: it commits a 64-action plan, executes the first 16 actions, then replans on the next observation.

**Training pipeline.** DAgger online training with **BFS oracle** supervision (`main.py --mode online`). Iteration 0 seeds the replay buffer with three oracle trajectories per training environment as an implicit BC warm-start; subsequent iterations alternate model rollouts → BFS oracle relabelling on the same seed → efficiency-filtered buffer insertion → AdamW gradient steps. The exponentially decaying β starts at 1.0; total budget ≈5.65M env steps with 30 episodes/iteration and 100 grad steps/iteration. Inference uses EMA-shadowed weights. The checkpoint loaded by this notebook is the iter-600 DAgger snapshot from a GPU-H200 run, with all optimiser/scheduler state stripped (the inference path only needs `ema_state_dict`).

### The objective under study

Fine-tuning runs on the planner's **own rollouts**. Each iteration collects
trajectories under the current policy and cuts them into `H`-step windows. Window
`i` is assigned the return `R_i` — the reward summed over exactly the `H` steps that
window trains on, *not* the episode total broadcast to every window — and receives a
scalar weight

```
A_i = clip( max(R_i, 0) / (µ_batch + ε),  c_min, c_max ),   [c_min, c_max] = [0.1, 5.0]
```

The training loss is the ELBO of the denoising objective with each window's term
scaled by `A_i`, treated as a constant under a stop-gradient. This is a **clipped
return ratio**, in the lineage of reward-weighted regression — it is *not* the
advantage-weighted regression weight, which subtracts a baseline and applies an
exponential temperature `exp(A/β)`. The paper's conclusions characterise this weight
function and do not carry over automatically to the exponential form, which was not
tested.

### The exact decomposition

Write `Ā` for the mean weight and `δ_i = A_i/Ā − 1`, so that `Σ_i δ_i = 0`. The
gradient then factors **exactly**, with no approximation:

```
∇L_RW  =  Ā · [  ∇L_BC   +   (1/B) Σ_i δ_i ∇ℓ_i  ]
                 ‾‾‾‾‾‾        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
                imitation      g_δ — the return's entire contribution
```

`Ā` is a scalar that rescales the step size and leaves the direction unchanged, so
everything the return does enters through `g_δ`, whose RMS scale is the weights'
coefficient of variation `CV_A = sqrt(B/ESS − 1)`. Both terms are measurable at a
single parameter point, and both diagnostics cost one batch and no accelerator.

### Three contributions (paper headline)

1. **Fine-tuning degrades the planner.** On Craftax Classic, across 25 conditions
   with three seeds each, **no condition recovers the checkpoint it started from**
   and the plain objective gives up **3.59 of 11.81** points. Collected return
   falls alongside held-out score (5.21 → 3.73 while eval goes 12.06 → 8.49), so
   this is not reward hacking under an evaluation mismatch. The conditions that
   lose least are the ones that barely train: LoRA (11.63), a hard KL trust region
   (11.53) and head-only updates (11.30).
2. **The return term is large and points away from imitation.** `CV_A` is **1.16**
   on Craftax Classic (effective sample size 437 of a 1024-window batch) and
   **2.68** on MiniHack (591 of 4608), so the weights do rank windows and the
   familiar explanation — that sparse returns fail to discriminate — is
   unavailable. Measured directly, `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine
   **0.02 ± 0.05**, against **0.893 ± 0.010** between two independent noise draws
   of `∇L_BC` itself. In `D = 9.33 × 10⁶` dimensions a random direction gives
   `3.3 × 10⁻⁴`, so `g_δ` is structure, not noise.
3. **But shrinking it makes things worse.** Advantage clipping confines the weights
   to [0.8, 1.2] and cuts the return term fivefold, to **0.097 ± 0.003**. It scores
   **5.06** on Craftax Classic — 3.16 below the unclipped baseline and second worst
   in the suite. A condition with almost no return term degrades *further* than one
   with a large one, so the return weighting is not what does the damage. What is
   left is the data: advantage clipping is also the nearest thing in the suite to
   unweighted training on the model's own rollouts, and the suite is ordered by how
   much plasticity each condition allows rather than by any property of the reward.

*(Live evaluation in Cell 5; imitation context in Cell 8; ablation evidence in Cells 9–10.)*

## Cell 4 — Load the pre-trained model

We load the architecture from `src/models/denoiser.py` using the project's own `make_model(cfg)` factory, then pour the EMA shadow weights from the stripped checkpoint into it. This is the same EMA-evaluation path that `main.py --mode inference` uses internally.

In [ ]:
import torch

# Load the project's authoritative config (the same defaults.yaml the
# training run used). All sampling / arch hyperparameters live there.
_overrides = {"device": INFERENCE_DEVICE} if INFERENCE_DEVICE else {}
cfg = load_config(cli_overrides=_overrides)
device = torch.device(cfg.device)
print(f"Inference device: {device}")
print(
    f"Model arch: n_embd={cfg.n_embd}, n_head={cfg.n_head}, "
    f"n_layer={cfg.n_layer}, n_global_tokens={cfg.n_global_tokens}, "
    f"seq_len={cfg.seq_len}"
)
print(
    f"Diffusion: K_eval={cfg.diffusion_steps_eval}, schedule={cfg.noise_schedule}, "
    f"remask={cfg.remask_strategy}, eta={cfg.eta}, T={cfg.temperature}, top_p={cfg.top_p}"
)

# Build the architecture and load the stripped EMA checkpoint. The HF repo
# ships an inference-only checkpoint that contains only the EMA shadow weights
# (≈21 MB) — no optimiser, scheduler, RNG, or curriculum state.
ckpt_path = os.path.join(SNAPSHOT_DIR, "checkpoint_inference.pth")
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f"Expected stripped checkpoint at {ckpt_path}. The HF repo must "
        f"contain checkpoint_inference.pth at its root."
    )

model = make_model(cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
ema_sd = ckpt["ema_state_dict"] if isinstance(ckpt, dict) and "ema_state_dict" in ckpt else ckpt
ema = ModelEMA(model, decay=cfg.ema_decay)
ema.load_state_dict(ema_sd)
ema.apply_to(model)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(
    f"Loaded EMA weights into {type(model).__name__} "
    f"({n_params:,} parameters ≈ {n_params / 1e6:.2f} M)."
)

## Cell 5 — Live inference on procedurally generated layouts ⭐

We now run the model live on `EPISODES_PER_ENV` episodes per environment, on the 4 in-distribution maps **and** the 3 out-of-distribution maps **and** the optional custom `.des` file. Each episode uses a per-`SEED` RNG so changing `SEED` in Cell 1 produces a fresh batch of procedurally generated layouts that the model has never trained or evaluated on.

We call the project's batched evaluator directly (`Evaluator._run_episodes_batched`), which runs all `EPISODES_PER_ENV` rollouts of a given environment in lockstep and batches every replanning forward pass through the GPU. Episodes that fail to construct (e.g. an invalid `.des` file) count as losses individually rather than crashing the pass.

In [ ]:
import time
from pathlib import Path

import numpy as np
import polars as pl

evaluator = Evaluator()


def run_live_eval(
    env_ids: list[str],
    n_episodes: int,
    seed: int,
    des_files: list[str] | None = None,
) -> dict[str, dict]:
    """Evaluate the model on each env id with seeds derived from *seed*.

    Mirrors ``Evaluator.evaluate`` but uses ``seed + ep`` so that changing
    the notebook ``SEED`` constant actually rotates the underlying
    procedural layouts.
    """
    targets: list[tuple[str, str | None]] = [(eid, None) for eid in env_ids]
    if des_files:
        for p in des_files:
            with open(p) as fh:
                targets.append((Path(p).stem, fh.read()))

    out: dict[str, dict] = {}
    for env_id, des_content in targets:
        seeds = [seed + ep for ep in range(n_episodes)]
        eps = evaluator._run_episodes_batched(
            model, env_id, n_episodes, cfg, device,
            seeds=seeds,
            des_content=des_content,
            blind_global=False,
        )
        wins = sum(1 for r in eps if r["won"])
        n = max(len(eps), 1)
        out[env_id] = {
            "win_rate": wins / n,
            "wins": wins,
            "avg_reward": sum(r["total_reward"] for r in eps) / n,
            "avg_steps": sum(r["steps"] for r in eps) / n,
            "n_episodes": len(eps),
        }
    return out


all_envs = list(ID_ENVS) + list(OOD_ENVS)
des_files = [CUSTOM_DES_FILE] if CUSTOM_DES_FILE else None
n_total = len(all_envs) * EPISODES_PER_ENV
if des_files:
    n_total += EPISODES_PER_ENV

print(
    f"Evaluating on {len(all_envs)} registry envs"
    + (f" + {len(des_files)} custom .des" if des_files else "")
    + f" × {EPISODES_PER_ENV} episodes (= {n_total} rollouts)..."
)
t0 = time.time()
live_results = run_live_eval(
    all_envs, EPISODES_PER_ENV, SEED, des_files=des_files,
)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed / max(n_total, 1):.2f}s per rollout).\n")

print(format_eval_results(live_results, label="Live ReMDM Inference"))

In [ ]:
# Tabulate and split-aware aggregate.
rows = []
for env_id, stats in live_results.items():
    if env_id in ID_ENVS:
        split = "ID"
    elif env_id in OOD_ENVS:
        split = "OOD"
    else:
        split = "custom"
    rows.append(
        {
            "split": split,
            "env": env_id,
            "win_rate": round(stats["win_rate"], 3),
            "avg_steps": round(stats["avg_steps"], 1),
            "avg_reward": round(stats["avg_reward"], 2),
            "n_episodes": stats["n_episodes"],
        }
    )
df_live = pl.DataFrame(rows).sort(["split", "env"])
print(df_live)

id_wr = float(np.mean([live_results[e]["win_rate"] for e in ID_ENVS if e in live_results]))
ood_wr = float(np.mean([live_results[e]["win_rate"] for e in OOD_ENVS if e in live_results]))
print(
    f"\nMean ID  win rate (this run): {id_wr:.2%}"
    f"   <- reference (paper Table 6, ReMDM DAgger, 50 episodes/env): 48.5%"
)
print(
    f"Mean OOD win rate (this run): {ood_wr:.2%}"
    f"   <- reference (paper Table 6, ReMDM DAgger): 4.7% mean OOD"
    f" (Room-Dark-15x15 12%, Corridor-R5 2%, MazeWalk-45x19 0%)"
)
print(
    "\nNote: small deviations from the reported numbers are expected -- this run "
    f"uses {EPISODES_PER_ENV} episodes per env (vs 50) and a different seed offset, "
    "so it samples a different (smaller) batch of procedurally generated layouts."
)

## Cell 6 — Visualise the agent's behaviour ⭐

We roll out a single episode on a representative ID environment and capture the dual-stream observations the model actually sees: the **9×9 local glyph crop** (centred on the agent `@`) and the **full 21×79 global dungeon map**. We display the start, mid, and end of the episode side-by-side.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

VIZ_ENV = ID_ENVS[2]  # MiniHack-Corridor-R2-v0 — visually richer than a single room

env = make_env(VIZ_ENV, None, cfg)
(local_obs, global_obs), _ = env.reset(seed=SEED)

trajectory_local: list[np.ndarray] = [local_obs.copy()]
trajectory_global: list[np.ndarray] = [global_obs.copy()]
actions_taken: list[int] = []
plan = None
step_in_plan = 0
total_reward = 0.0
won = False

for step in range(200):
    if step_in_plan == 0 or step_in_plan >= cfg.replan_every:
        # Replan: full ReMDM denoising over the current local + global obs.
        local_t = torch.from_numpy(local_obs).long().unsqueeze(0).to(device)  # [1, 9, 9]
        glb_t = torch.from_numpy(global_obs).long().unsqueeze(0).to(device)  # [1, 21, 79]
        plan = remdm_sample(
            model, local_t, glb_t, cfg, device,
            physics_aware=False, blind_global=False,
        )[0].cpu().numpy()  # [seq_len]
        step_in_plan = 0

    action = int(plan[step_in_plan])
    step_in_plan += 1
    actions_taken.append(action)
    (local_obs, global_obs), reward, term, trunc, info = env.step(action)
    total_reward += reward
    trajectory_local.append(local_obs.copy())
    trajectory_global.append(global_obs.copy())
    if info.get("won"):
        won = True
    if term or trunc:
        break

env.close()
n_steps = len(actions_taken)
print(
    f"{VIZ_ENV} (seed={SEED}): {n_steps} steps, won={won}, total_reward={total_reward:.2f}"
)

In [ ]:
snapshots = [
    ("Start", 0),
    ("Mid", n_steps // 2),
    ("End", n_steps),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for col, (label, idx) in enumerate(snapshots):
    axes[0, col].imshow(trajectory_local[idx], cmap="viridis")
    axes[0, col].set_title(f"{label} — local 9×9 (step {idx})")
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    axes[1, col].imshow(trajectory_global[idx], cmap="viridis")
    axes[1, col].set_title(f"{label} — global 21×79 (step {idx})")
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.suptitle(
    f"Dual-stream observations during a live rollout on {VIZ_ENV}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(
    "Each colour above is a NetHack glyph ID (walls, floor, agent @, staircase >, "
    "doors +, etc.). The local crop is the 9×9 window the local CNN stream sees; "
    "the global stream sees the full 21×79 grid and contributes the auxiliary "
    "staircase-coordinate prediction through the goal head."
)

## Cell 7 — Visualise the ReMDM denoising process ⭐

This is the heart of the method. Given a single observation, the planner starts from a sequence of 64 `[MASK]` tokens and over `K = diffusion_steps_eval = 10` reverse-diffusion steps it (a) predicts an action distribution at every position, (b) commits the highest-confidence positions via MaskGIT-style progressive unmasking, and (c) **stochastically remasks** previously committed positions whose confidence is low so they can be re-decoded later. We surface the per-step state by calling `remdm_sample(..., return_analytics=True)`, which returns the full denoising trajectory and per-step diagnostics.

In [ ]:
# Use a fresh observation from the same env so the visualisation stays focused
# on the token-level denoising process, not on action quality.
env = make_env(ID_ENVS[2], None, cfg)
(local_obs, global_obs), _ = env.reset(seed=SEED)
env.close()

local_t = torch.from_numpy(local_obs).long().unsqueeze(0).to(device)
glb_t = torch.from_numpy(global_obs).long().unsqueeze(0).to(device)

seq, path_per_step, conf_track, masked_track = remdm_sample(
    model, local_t, glb_t, cfg, device,
    physics_aware=False, blind_global=False,
    return_analytics=True,
)
K = len(path_per_step)
seq_len = cfg.seq_len
mask_token = cfg.mask_token

# Build a [K+1, seq_len] visualisation: row 0 = the all-MASK initial state,
# row k = the sequence after denoising step k.
vis = np.full((K + 1, seq_len), mask_token, dtype=np.int64)
for k, state in enumerate(path_per_step):
    vis[k + 1] = state
# Encode masked positions as -1 so they get a distinct colour band.
display_grid = np.where(vis == mask_token, -1, vis).astype(float)

print(
    f"Denoising trajectory: K={K} steps, seq_len={seq_len}, "
    f"final masked-token count={int((seq[0] == mask_token).sum().item())} (must be 0)"
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 6),
    gridspec_kw={"height_ratios": [3, 1]},
)

im = ax1.imshow(
    display_grid, aspect="auto", cmap="viridis",
    vmin=-1, vmax=cfg.action_dim - 1,
    interpolation="nearest",
)
ax1.set_yticks(range(K + 1))
ax1.set_yticklabels(["init"] + [f"k={k+1}" for k in range(K)])
ax1.set_xlabel("token position (0..63)")
ax1.set_ylabel("denoising step")
ax1.set_title(
    f"ReMDM denoising trajectory ({K} steps) — "
    "dark band (-1) = masked, colours = committed action ids 0..11"
)
cbar = plt.colorbar(im, ax=ax1, ticks=[-1] + list(range(cfg.action_dim)))
cbar.set_label("token id (-1 = MASK)")

ax2.plot(range(1, K + 1), conf_track, marker="o", label="avg confidence (committed tokens)")
ax2.plot(
    range(1, K + 1),
    [m / seq_len for m in masked_track],
    marker="s",
    label="fraction still masked",
)
ax2.set_xlabel("denoising step k")
ax2.set_ylabel("value")
ax2.set_xticks(range(1, K + 1))
ax2.legend(loc="upper right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final committed plan (first 16 actions): {seq[0, :16].tolist()}")
print(
    "Read the heatmap top-to-bottom: row 0 is fully masked, the next rows commit "
    "high-confidence positions while occasionally re-masking low-confidence ones "
    "(the ReMDM 'conf' strategy with eta=0.15), and the last row is the fully "
    "committed plan that gets executed for the next 16 env steps before replanning."
)

## Cell 8 — Imitation learning context (baselines)

The table below reports this project's MiniHack baseline runs, under the **imitation protocol**: 50 episodes per environment at inference settings. Baseline rows are means ± std over 3 training seeds; the two ReMDM rows are a single evaluation seed and are read directly from the shipped `results/inference/` artefacts.

**Provenance note.** The current paper carries the baseline *configurations* (Table 11, "MiniHack baseline hyperparameters") but no baseline *results* table — that comparison was cut when the paper narrowed to the fine-tuning result. The model-free and offline-BC rows below are therefore project context rather than a paper number. The two ReMDM rows *are* paper numbers: the DAgger row is paper Table 6.

The four model-free rows are included for **one purpose**: to establish that these layouts are not solvable from scratch within our compute budget, which is why we train by imitation. They use default hyperparameters and are **not a matched comparison**.

**This is not an architecture result.** Any gap to the offline CNN+MLP and Decision Transformer confounds four design differences: an auxiliary staircase-prediction head the baselines lack, ten denoising passes per decision against one, chunked replanning against greedy per-step selection, and untuned baselines.

**Two evaluation protocols appear in this work and their numbers are not interchangeable.** The direct evaluation of the DAgger checkpoint (this cell, 50 episodes per environment) gives **48.5%** in-distribution — this is paper Table 6. The ablation-harness protocol (Cells 9–10) runs 20 episodes per environment per seed inside the training loop at its own sampling settings, and gives the same checkpoint **47.5%**. The two agree closely here, but no claim crosses between them.

In [ ]:
import polars as pl

# Baseline runs under the imitation protocol: 50 episodes per environment at
# inference settings. Baseline rows are means +/- std over 3 training seeds.
#
# PROVENANCE: the current paper keeps the baseline *configurations* (Table 11)
# but no baseline *results* table, so the first six rows are project context,
# not paper numbers. The two ReMDM rows are read from the shipped
# results/inference/*.json artefacts; the DAgger row is paper Table 6.
baselines = [
    {"method": "PPO",                            "params": "~0.4 M", "id_win_pct": "0.3 +/- 0.5",  "id_steps":  "396.8 +/- 10.1", "ood_win_pct": "0.0 +/- 0.0", "ood_steps": "767.6 +/- 6.0"},
    {"method": "A2C",                            "params": "~0.4 M", "id_win_pct": "2.8 +/- 2.3",  "id_steps":  "392.9 +/- 8.1",  "ood_win_pct": "0.4 +/- 0.8", "ood_steps": "763.8 +/- 5.0"},
    {"method": "DQN",                            "params": "~0.4 M", "id_win_pct": "3.8 +/- 1.2",  "id_steps":  "388.2 +/- 5.2",  "ood_win_pct": "1.3 +/- 0.2", "ood_steps": "762.7 +/- 1.3"},
    {"method": "PPO-RNN",                        "params": "~0.5 M", "id_win_pct": "4.7 +/- 2.9",  "id_steps":  "385.4 +/- 13.2", "ood_win_pct": "0.0 +/- 0.0", "ood_steps": "766.7 +/- 3.1"},
    {"method": "CNN+MLP (Offline BC)",           "params": "~0.6 M", "id_win_pct": "6.3 +/- 2.5",  "id_steps":  "378.8 +/- 6.4",  "ood_win_pct": "0.7 +/- 0.7", "ood_steps": "764.7 +/- 2.0"},
    {"method": "Decision Transformer (Off. BC)", "params": "~1.0 M", "id_win_pct": "9.3 +/- 2.1",  "id_steps":  "160.7 +/- 4.2",  "ood_win_pct": "1.8 +/- 0.4", "ood_steps": "196.8 +/- 0.6"},
    {"method": "ReMDM (Offline BC)",             "params": "~5.2 M", "id_win_pct": "73.0",         "id_steps":  "107.1",          "ood_win_pct": "6.0",         "ood_steps": "418.5"},
    {"method": "ReMDM (DAgger)",                 "params": "~5.2 M", "id_win_pct": "48.5",         "id_steps":  "173.9",          "ood_win_pct": "4.7",         "ood_steps": "426.2"},
]
df_baselines = pl.DataFrame(baselines)
with pl.Config(tbl_rows=10, fmt_str_lengths=40):
    print(df_baselines)
print(
    "\nThe four model-free rows are NOT a matched comparison -- they use default "
    "hyperparameters and are reported only to establish that these layouts are not "
    "solvable from scratch within our budget (0.3-4.7% ID after 5M env steps), "
    "which is why we train by imitation.\n"
    "\nThe DAgger row (48.5% ID / 4.7% OOD) is paper Table 6, and it is the "
    "checkpoint every ablation condition in Cells 9-10 fine-tunes. The planner "
    "retains little zero-shot transfer to held-out layouts, so the paper makes no "
    "structural generalisation claim."
)

## Cell 9 — RL fine-tuning ablation findings (pre-computed)

On top of the DAgger checkpoint, we ran a **25-condition suite** (`experiments/rl_finetuning/run_ablations.py --all`) that asks what return-weighted ELBO fine-tuning of the diffusion planner actually does. The 25 conditions are organised into four mechanism groups (A: regularisation, B: training signal, C: architectural freezing, D: data quality / weight computation) plus a `baseline_rl` reference, all initialised from the same checkpoint and run for 500 iterations with 3 seeds. All numbers here use the **ablation-harness protocol**, under which the pretrained checkpoint scores **47.5%** — against 48.5% for the direct evaluation in Cell 8.

**In MiniHack the objective takes the checkpoint from 47.5% to 43.8% (±6.1).** Three conditions finish nominally above the checkpoint — `head_only` 49.6%, `gradient_surgery` 48.8%, `layer_ablation_top1` 48.8% — but the best-versus-baseline difference has a bootstrap interval of **[−2.1, +13.7] points at p = 0.40**, so the paper does not read them as improvements.

**MiniHack is not a second confirmation, and the paper says so.** The effect here is small relative to a seed standard deviation of 6.1 points. It is reported because it is the same suite run identically and its ordering matches Craftax Classic, where the effect is large and unambiguous (11.81 → 8.22, no condition above the checkpoint). The claim rests on Craftax Classic.

**The reward pathway is not the explanation.** `CV_A` averages **2.68** here (effective sample size 591 of a 4,608-window batch) against 1.16 on Craftax Classic, so in both environments the weights rank windows within a batch and the familiar explanation that sparse returns fail to discriminate is unavailable. The direct measurement of the return term was only possible on Craftax Classic — MiniHack needs the NetHack Learning Environment, which would not build on the machine used for that measurement — and there it gives `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine `0.02 ± 0.05`. The control that settles it is also Craftax-side: advantage clipping cuts the return term fivefold and scores **3.16 points worse**, so the return weighting is not what does the damage.

**What orders the outcome is permitted displacement.** Final KL divergence from the checkpoint correlates with final score at Spearman **−0.56** here and **−0.79** on Craftax Classic (−0.56 and −0.75 with LoRA excluded, which matters because the drift probe reads LoRA's frozen base weights and records near-zero). The sign is the same in both environments now: more displacement, lower score. Group C (architectural freezing) has the highest group mean here (**44.0%**) and Group A the tightest spread (std 0.95 points) — freezing neither destabilises the model nor helps it past the checkpoint.

**Group-level summary** (mean ID win rate over each group, Table 7 from the paper):

| Group | N | Mean | Δ from Baseline RL | Best | Worst | Std |
|---|---|---|---|---|---|---|
| Pretrained checkpoint | – | **47.50** | – | – | – | – |
| Baseline RL | 1 | 43.75 | – | – | – | 0.00 |
| A (Regularisation) | 6 | 42.78 | −0.97 | 44.17 | 41.25 | 0.95 |
| B (Training signal) | 7 | 38.57 | −5.18 | 48.75 | 12.08 | 11.64 |
| C (Architectural freezing) | 7 | 43.99 | +0.24 | 49.58 | 39.17 | 4.08 |
| D (Data quality / weights) | 4 | 38.02 | −5.73 | 44.17 | 32.08 | 4.28 |

Every group mean is below the pretrained checkpoint, in both environments.

Group B degrades with high variance (std 11.64), driven by the worst condition in the suite: **normalised advantages** at **12.1%** (3.73 on Craftax Classic). It alone mean-centres the weights, making roughly half of them **negative** and violating the non-negativity assumption (A1) the decomposition needs. A negative weight on a cross-entropy term is gradient *ascent* on that sequence's likelihood — unbounded and without a trust region — and its drift lands two to four orders of magnitude beyond any other condition. Supplying the repulsive direction a policy gradient would have does not rescue the objective; it destroys it.

**On its own terms.** Craftax Classic collected return falls (5.21 → 3.73) alongside eval score, so the objective is not quietly succeeding at what it optimises. On MiniHack collected return is flat (−0.53 → −0.34, no trend across quartiles) while the training-window win rate rises from 6.3% to 13.5% and eval is flat. Neither environment looks like reward hacking under an evaluation mismatch.

**Statistics.** With 3 seeds per arm, an exact two-sided permutation test enumerates C(6,3) = 20 relabellings, so the smallest attainable p-value is 2/20 = **0.10** — a reported p = 0.10 is the most extreme outcome the design can produce, not a null result. The shipped `significance_test.txt` compares the condition furthest from baseline (`normalized_adv`) and reports p at that floor with a bootstrap CI of **[−39.6, −23.8] points**; the paper's **[−2.1, +13.7]** interval is the different, best-versus-baseline comparison. Cell 10 prints the shipped file verbatim so the two are not confused.

Each condition also collects gradient-alignment, representation-drift, CKA-similarity, and per-t-bin gradient-norm diagnostics. All training was done **offline** on the GPU-H200 machine — this notebook only displays the saved figures (downloaded with the HF snapshot).

In [ ]:
from IPython.display import Image, display

assets_dir = os.path.join(SNAPSHOT_DIR, "ablation_assets")
if not os.path.isdir(assets_dir):
    raise FileNotFoundError(
        f"Expected ablation assets at {assets_dir}. The HF repo must contain "
        "an ablation_assets/ directory with the pre-computed PNGs and CSVs."
    )

figures = [
    (
        "final_score_comparison.png",
        "Final ID win rate per condition, sorted, against the pretrained "
        "checkpoint at 47.5% under the harness protocol. Baseline RL falls to "
        "43.8% (+/-6.1). Three conditions -- head_only 49.6, gradient_surgery "
        "48.8, layer_ablation_top1 48.8 -- finish nominally above the "
        "checkpoint, but the best-versus-baseline bootstrap interval is "
        "[-2.1, +13.7] points at p = 0.40, so they are not read as "
        "improvements. The worst condition is normalized_adv at 12.1%, and it "
        "collapses because it mean-centres the weights and makes half of them "
        "negative -- gradient ascent on below-average rollouts -- not because "
        "the reward signal is uninformative. CV_A here is 2.68: the weights do "
        "rank windows.",
    ),
    (
        "group_comparison.png",
        "Score distribution by intervention group, all below the pretrained "
        "47.5%. Group C (architectural freezing) has the highest mean at 44.0, "
        "Group A (regularisation) the tightest spread at 42.8 +/- 0.95, and "
        "Groups B (38.6) and D (38.0) sit lowest. Group B's spread (std 11.6) "
        "is carried by the normalized_adv collapse. Freezing parameter subsets "
        "neither destabilises the model nor carries it past the checkpoint.",
    ),
    (
        "score_delta_over_baseline_rl.png",
        "Sorted improvement vs the baseline_rl condition. A handful of "
        "conditions beat baseline return-weighted ELBO fine-tuning; on Craftax "
        "Classic, where the effect is unambiguous, none of the 25 reaches the "
        "checkpoint it started from. What orders this ranking is how far each "
        "condition is permitted to displace the parameters (Spearman -0.56 "
        "between final KL and final score here, -0.79 on Craftax Classic), not "
        "how the return is turned into a weight.",
    ),
    (
        "per_env_delta.png",
        "Per-environment change from the pretrained checkpoint. The easy rooms "
        "sit near ceiling and move little; the losses concentrate in the "
        "layouts that need longer routing. This mirrors Craftax Classic, where "
        "deep tech-tree achievements are the first to go (tier 3 falls 23% to "
        "10%) while shallow behaviours hold.",
    ),
    (
        "gradient_alignment.png",
        "Cosine similarity between the return-weighted RL loss and the BC loss "
        "gradients over fine-tuning. NOTE: the two gradients are taken at "
        "*different parameter points*, so this is a retention diagnostic only "
        "-- it confounds weighting with drift and is unrelated to the "
        "single-parameter-point measurement of ||g_delta||/||grad L_BC|| = "
        "0.49 +/- 0.01 at cosine 0.02 +/- 0.05 quoted above.",
    ),
]

for fname, caption in figures:
    path = os.path.join(assets_dir, fname)
    if os.path.exists(path):
        print(f"\n=== {fname} ===\n{caption}\n")
        display(Image(filename=path))
    else:
        print(f"\n[missing figure: {fname}]")

## Cell 10 — Ablation results tables

The tables below are loaded directly from the saved CSV outputs of the ablation pipeline. `main_results.csv` is the canonical sortable score table; `hypothesis_verdict.csv` attaches each condition to the mechanism hypothesis it tests, so the verdict column reads as a direct answer to *what does return-weighted ELBO fine-tuning of the diffusion planner actually do?*

**Read the `Verdict` column carefully.** It is scored against **baseline RL**, not against the pretrained checkpoint. A row marked `IMPROVEMENT` beat the plain return-weighted objective; with three exceptions here (and none at all on Craftax Classic) it did not beat the checkpoint it started from. `Delta_Pretrained` is the column that answers that, and it is negative for 22 of the 25 conditions in MiniHack and for all 25 on Craftax Classic.

The collective answer across all 25 conditions is that the outcome does not track how the return is turned into a weight — the Craftax-side control cuts the return term fivefold and scores 3.16 *worse* — but does track how far each condition is permitted to displace the parameters. What that leaves pointing at is the data: fine-tuning on the model's own rollouts, with the return weighting a large but incidental passenger.

In [ ]:
import polars as pl

main_csv = os.path.join(assets_dir, "main_results.csv")
verdicts_csv = os.path.join(assets_dir, "hypothesis_verdict.csv")

df_main = pl.read_csv(main_csv).sort("Score", descending=True)
print("=== main_results.csv (sorted by Score, descending) ===")
with pl.Config(tbl_rows=30):
    print(df_main)

In [ ]:
df_verdicts = pl.read_csv(verdicts_csv).sort("Delta_Baseline", descending=True)
print("=== hypothesis_verdict.csv (sorted by improvement over baseline_rl) ===")
with pl.Config(tbl_rows=30, fmt_str_lengths=80):
    print(df_verdicts)

## Cell 11 — Conclusions

**Empirical findings.**

1. **The planner learns these layouts; the model-free baselines do not.** The DAgger-trained ReMDM planner reaches **48.5% in-distribution** over 50 episodes per environment (paper Table 6), and the offline-BC variant **73.0%**, against 0.3–4.7% for PPO, A2C, DQN and PPO-RNN trained from scratch. Those four are **not a matched comparison** — they use default hyperparameters and are reported only to establish that the layouts are not solvable from scratch within our budget. Any gap to the offline CNN+MLP and Decision Transformer confounds an auxiliary staircase-prediction head the baselines lack, ten denoising passes per decision against one, chunked replanning against greedy per-step selection, and untuned baselines. **This is not an architecture result**, and the current paper carries these baselines only as configurations (Table 11), not as a results table.

2. **Zero-shot OOD transfer is weak, and we make no structural transfer claim.** The DAgger planner reaches **4.7%** mean win rate on the 3 held-out OOD maps against 48.5% in distribution (Room-Dark-15x15 12%, Corridor-R5 2%, MazeWalk-45x19 0%). The offline-BC variant reaches 6.0%. What transfer exists is to a partially observed room, not to longer routing or larger mazes.

3. **Return-weighted ELBO fine-tuning degrades the planner — and the reward signal is not why.** In MiniHack the objective takes the checkpoint from **47.5% to 43.8% (±6.1)**; three conditions finish nominally above it, but the best-versus-baseline interval is [−2.1, +13.7] at p = 0.40, so they are not read as improvements. On Craftax Classic, where the effect is large and unambiguous, **no condition of the 25 recovers the checkpoint** and the plain objective gives up **3.59 of 11.81** points. The explanation is *not* that sparse returns fail to discriminate: `CV_A` is **2.68** here and **1.16** on Craftax Classic, and the return term itself measures `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine **0.02 ± 0.05** — large and pointing away from imitation, against 0.893 ± 0.010 for `∇L_BC` against itself. **But shrinking it makes things worse.** Advantage clipping cuts the return term fivefold, to 0.097 ± 0.003, and scores **3.16 points below** the unclipped baseline. A condition with almost no return term degrades further than one with a large one.

4. **What is left is the data.** The suite is ordered by plasticity — Spearman **−0.56** here and **−0.79** on Craftax Classic between final KL and final score — and advantage clipping, the nearest thing in the suite to unweighted training on the model's own rollouts, sits near the bottom. Together these point at fine-tuning on self-generated rollouts as the damaging ingredient, with the return weighting neither causing the harm nor preventing it. The paper is explicit that this is an inference from a near-substitute: the unweighted-ELBO-on-all-rollouts arm was not run.

5. **It is not reward hacking under an evaluation mismatch.** On Craftax Classic collected return falls from **5.21 to 3.73** over the same 500 iterations in which eval score falls from 12.06 to 8.49. On MiniHack collected return is flat (−0.53 → −0.34) while the training-window win rate rises from 6.3% to 13.5% and eval is flat. The objective is not quietly succeeding at what it optimises.

**Why the sign constraint is not the whole story.** The objective is a bound only for `A_i ≥ 0`, so it can re-rank sampled behaviour but never push mass away from it, which caps how much *improvement* re-ranking can deliver. That does not explain *degradation*, and it cannot be the whole story, because the same non-negativity holds of reward-weighted and advantage-weighted regression, which work elsewhere. The difference we can point to is the data: those methods are usually applied to a fixed dataset rather than to rollouts fed back through a denoising objective at every step.

**A second, separate cost of the surrogate.** Not offered as evidence for the above: the ELBO surrogate is badly conditioned across diffusion time, and markedly worse on the harder environment. On Craftax Classic the gradient norm in the top third of the `t` range is **2.5×** the bottom third with low-`t`/high-`t` cosine similarity **0.06**; on MiniHack the ratio is **1.14** and the cosine **0.52**. Repairing the conditioning does not recover the checkpoint — on Craftax Classic `low_t` (8.12) and `t_curriculum` (8.41) sit at the baseline of 8.22 rather than above it.

**What these results do not establish.** MiniHack is **not a second confirmation**: the effect here is small against a 6.1-point seed standard deviation, and the claim rests on Craftax Classic. There is no unweighted-ELBO-on-all-rollouts arm and no continued-DAgger arm. The gradient measurement is at one parameter point, on Craftax Classic only — MiniHack would need the NetHack Learning Environment, which would not build on the machine used for that measurement. The null characterises the clipped return ratio, not the exponential advantage weight `exp(A/β)`. And the gradient-alignment cosine logged by the training harness takes its two gradients at *different* parameter points, so it is a **retention diagnostic only**.

**Two cheap diagnostics, worth taking first.** For anyone reaching for this objective: `CV_A` comes free from an effective-sample-size counter and says whether the returns rank anything at all. The ratio `‖g_δ‖/‖∇L_BC‖` costs two backward passes and says how much of the update the reward is responsible for. Ours said 0.49, which looked like a mechanism until the clipping control said otherwise.

**Open problem.** Per-step formulations over the denoising chain (d1, DiffPO) admit **signed** advantages, and so change the object being optimised rather than the weighting inside a regression. The obstacle for planners like ours is that those estimators assume monotone unmasking, which ReMDM's inference-time remasking violates. What these results argue for is an estimator that tolerates remasking *and* admits a repulsive direction; **categorical flow matching** (Campbell et al., 2024) is one route to the first half.